# Lab 06 — 00 Source Preparation (External Tables Edition)

**Dataset:** Synthea Healthcare  
**Architecture:** External raw volume → Gold external Delta tables

## Purpose

This notebook prepares and validates the raw Synthea source data required by
`lab_06_gold_analytics_external`.

It intentionally separates **source preparation** from the Gold external-table
implementation:

```text
External Unity Catalog Volume
        ↓
Synthea CSV source files
        ↓
00_source_preparation
        ↓
01_dimensions
        ↓
02_fact_encounters
        ↓
03_fact_conditions
        ↓
04_aggregations
        ↓
05_register_shared_tables
        ↓
06_alert_metrics
        ↓
07_validation
```

### What this notebook does

1. Reads development parameters.
2. Resolves the Lab 06 repository configuration.
3. Confirms the source Unity Catalog volume exists and is **EXTERNAL**.
4. Creates the source/reference/landing folders if needed.
5. Downloads the official Synthea 1K CSV sample archive.
6. Extracts CSV files directly into the external volume.
7. Verifies the six Lab 06 source datasets.
8. Profiles source row counts and schemas.
9. Validates required columns.
10. Performs basic key and relationship checks.
11. Stages reference/master CSV files.
12. Produces a final PASS/FAIL validation matrix.

> This notebook does **not** create Gold dimensions, facts, aggregates, or external
> Delta tables. Those responsibilities remain in the later Lab 06 notebooks and
> `src/external_tables.py`.


## 1. Development parameters

In [0]:
def ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(
    name: str,
    default: str,
    choices: list[str],
    label: str,
) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget("schema", "parvinbadalov", "02 Schema")
ensure_text_widget(
    "volume_name",
    "lab06_gold_analytics",
    "03 Source external volume",
)
ensure_text_widget(
    "synthea_url",
    (
        "https://raw.githubusercontent.com/"
        "synthetichealth/synthea-sample-data/main/downloads/"
        "synthea_sample_data_csv_nov2021.zip"
    ),
    "04 Synthea ZIP URL",
)
ensure_dropdown_widget(
    "refresh_source",
    "false",
    ["false", "true"],
    "05 Refresh source files",
)
ensure_dropdown_widget(
    "run_validation",
    "true",
    ["true", "false"],
    "06 Run validation",
)

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()
synthea_url = dbutils.widgets.get("synthea_url").strip()
refresh_source = dbutils.widgets.get("refresh_source").lower() == "true"
run_validation = dbutils.widgets.get("run_validation").lower() == "true"

print(f"Catalog          : {catalog}")
print(f"Schema           : {schema}")
print(f"Source volume    : {volume_name}")
print(f"Refresh source   : {refresh_source}")
print(f"Run validation   : {run_validation}")

## 2. Resolve shared Lab 06 configuration

The notebook first tries to reuse `src.config.Lab06Config`.  
If the external-table branch changes that class, the source paths still fall
back to the same parameterized `/Volumes/<catalog>/<schema>/<volume>/...` layout.


In [0]:
import sys
from pathlib import Path

current_dir = Path.cwd()
lab_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

config = None

try:
    from src.config import Lab06Config

    try:
        config = Lab06Config(
            catalog=catalog,
            schema=schema,
            volume_name=volume_name,
        )
    except TypeError:
        # The external-table edition may evolve Lab06Config.
        # Source preparation can still use the standard UC Volume layout.
        config = None
except Exception as exc:
    print(f"Shared config import not used: {type(exc).__name__}: {exc}")

volume_fqn = f"{catalog}.{schema}.{volume_name}"

if config is not None and hasattr(config, "volume_path"):
    volume_path = config.volume_path
else:
    volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"

source_csv_path = (
    config.source_csv_path
    if config is not None and hasattr(config, "source_csv_path")
    else f"{volume_path}/source/csv"
)

reference_path = (
    config.reference_path
    if config is not None and hasattr(config, "reference_path")
    else f"{volume_path}/reference"
)

encounter_landing_path = (
    config.encounter_landing_path
    if config is not None and hasattr(config, "encounter_landing_path")
    else f"{volume_path}/landing/encounters"
)

print(f"Repository root    : {lab_root}")
print(f"Volume FQN         : {volume_fqn}")
print(f"Volume path        : {volume_path}")
print(f"CSV source path    : {source_csv_path}")
print(f"Reference path     : {reference_path}")
print(f"Encounter landing  : {encounter_landing_path}")

## 3. Validate the external Unity Catalog volume

In [0]:
volume_description_df = spark.sql(
    f"DESCRIBE VOLUME {volume_fqn}"
)

display(volume_description_df)

volume_row = volume_description_df.first().asDict()
volume_type = str(volume_row.get("volume_type", "")).upper()
storage_location = volume_row.get("storage_location")

if volume_type != "EXTERNAL":
    raise ValueError(
        f"{volume_fqn} is {volume_type or 'UNKNOWN'}; "
        "Lab 06 source preparation expects an EXTERNAL volume."
    )

if not storage_location:
    raise ValueError(
        f"{volume_fqn} does not expose an external storage location."
    )

print("External volume validation passed.")
print(f"Storage location: {storage_location}")

## 4. Prepare source directories

In [0]:
for path in [
    source_csv_path,
    reference_path,
    encounter_landing_path,
]:
    dbutils.fs.mkdirs(path)

print("Lab 06 source directories are ready.")

## 5. Inspect the current source state

By default, an existing valid source is reused.  
Set **Refresh source = true** only when you intentionally want to download and
replace the raw Synthea CSV files.


In [0]:
REQUIRED_FILES = {
    "patients": "patients.csv",
    "encounters": "encounters.csv",
    "providers": "providers.csv",
    "organizations": "organizations.csv",
    "payers": "payers.csv",
    "conditions": "conditions.csv",
}

try:
    current_source_files = {
        item.name.rstrip("/")
        for item in dbutils.fs.ls(source_csv_path)
    }
except Exception:
    current_source_files = set()

required_file_names = set(REQUIRED_FILES.values())
missing_before_download = sorted(
    required_file_names - current_source_files
)

print(f"Existing files       : {len(current_source_files)}")
print(f"Missing required     : {len(missing_before_download)}")

if missing_before_download:
    print("Missing:", ", ".join(missing_before_download))
else:
    print("All required Lab 06 CSV files already exist.")

## 6. Download and extract Synthea when required

In [0]:
import io
import os
import shutil
import zipfile

import requests

should_download = refresh_source or bool(missing_before_download)

if should_download:
    print(f"Downloading: {synthea_url}")

    response = requests.get(
        synthea_url,
        timeout=120,
        headers={"User-Agent": "Databricks-Lab06-External/1.0"},
    )
    response.raise_for_status()

    archive_size_mb = len(response.content) / (1024 * 1024)
    print(f"Archive downloaded: {archive_size_mb:.2f} MB")

    copied_files = []

    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        csv_members = [
            member
            for member in archive.namelist()
            if member.startswith("csv/")
            and member.lower().endswith(".csv")
        ]

        if not csv_members:
            raise ValueError(
                "No CSV files were found in the Synthea archive."
            )

        for member in sorted(csv_members):
            filename = os.path.basename(member)
            target_file = f"{source_csv_path}/{filename}"

            with archive.open(member) as source_handle:
                with open(target_file, "wb") as target_handle:
                    shutil.copyfileobj(
                        source_handle,
                        target_handle,
                    )

            copied_files.append(filename)

    print(f"Copied {len(copied_files)} CSV files to:")
    print(source_csv_path)
else:
    print(
        "Source download skipped: all required files exist "
        "and refresh_source=false."
    )

## 7. Verify the six required Lab 06 datasets

In [0]:
available_files = {
    item.name.rstrip("/")
    for item in dbutils.fs.ls(source_csv_path)
}

source_file_validation = [
    (
        dataset,
        filename,
        "PASS" if filename in available_files else "FAIL",
    )
    for dataset, filename in REQUIRED_FILES.items()
]

source_file_validation_df = spark.createDataFrame(
    source_file_validation,
    ["dataset", "file_name", "status"],
)

display(source_file_validation_df)

missing_files = sorted(
    required_file_names - available_files
)

if missing_files:
    raise FileNotFoundError(
        "Missing required Synthea source files: "
        + ", ".join(missing_files)
    )

print("All six required Synthea source files are available.")

## 8. Load and profile source datasets

`inferSchema` is used only for source exploration.  
The Gold notebooks remain responsible for explicit business transformations and
target schemas.


In [0]:
source_dfs = {}
inventory_rows = []

for dataset, filename in REQUIRED_FILES.items():
    path = f"{source_csv_path}/{filename}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

    source_dfs[dataset] = df

    inventory_rows.append(
        (
            dataset,
            filename,
            df.count(),
            len(df.columns),
        )
    )

source_inventory_df = spark.createDataFrame(
    inventory_rows,
    ["dataset", "file_name", "row_count", "column_count"],
)

display(source_inventory_df.orderBy("dataset"))

### Expected Synthea 1K sample counts

The November 2021 Synthea sample normally contains approximately:

| Dataset | Rows |
|---|---:|
| patients | 1,163 |
| encounters | 61,459 |
| providers | 5,056 |
| organizations | 1,127 |
| payers | 10 |
| conditions | 38,094 |

The notebook does not fail solely because a future source revision has different
counts; structural and key validations are more important.


## 9. Validate required source columns

In [0]:
REQUIRED_COLUMNS = {
    "patients": {
        "Id",
        "BIRTHDATE",
        "FIRST",
        "LAST",
    },
    "encounters": {
        "Id",
        "START",
        "STOP",
        "PATIENT",
        "ORGANIZATION",
        "PROVIDER",
        "PAYER",
    },
    "providers": {
        "Id",
        "ORGANIZATION",
        "NAME",
    },
    "organizations": {
        "Id",
        "NAME",
    },
    "payers": {
        "Id",
        "NAME",
    },
    "conditions": {
        "START",
        "PATIENT",
        "CODE",
        "DESCRIPTION",
    },
}

column_validation_rows = []
column_failures = []

for dataset, required_columns in REQUIRED_COLUMNS.items():
    actual_columns = set(source_dfs[dataset].columns)
    missing_columns = sorted(required_columns - actual_columns)

    status = "PASS" if not missing_columns else "FAIL"

    column_validation_rows.append(
        (
            dataset,
            status,
            ", ".join(missing_columns),
        )
    )

    if missing_columns:
        column_failures.append(
            f"{dataset}: {', '.join(missing_columns)}"
        )

column_validation_df = spark.createDataFrame(
    column_validation_rows,
    ["dataset", "status", "missing_columns"],
)

display(column_validation_df.orderBy("dataset"))

if column_failures:
    raise ValueError(
        "Required source-column validation failed: "
        + " | ".join(column_failures)
    )

print("Required source-column validation passed.")

## 10. Validate source keys and encounter timestamps

In [0]:
from pyspark.sql import functions as F

key_checks = []

for dataset in [
    "patients",
    "encounters",
    "providers",
    "organizations",
    "payers",
]:
    df = source_dfs[dataset]

    if "Id" not in df.columns:
        key_checks.append(
            (dataset, None, None, None, "FAIL")
        )
        continue

    metrics = (
        df.agg(
            F.count("*").alias("row_count"),
            F.countDistinct("Id").alias("distinct_id_count"),
            F.sum(
                F.when(
                    F.col("Id").isNull()
                    | (F.trim(F.col("Id").cast("string")) == ""),
                    1,
                ).otherwise(0)
            ).alias("null_or_blank_ids"),
        )
        .first()
    )

    row_count = int(metrics["row_count"])
    distinct_ids = int(metrics["distinct_id_count"])
    null_ids = int(metrics["null_or_blank_ids"] or 0)

    status = (
        "PASS"
        if row_count == distinct_ids and null_ids == 0
        else "FAIL"
    )

    key_checks.append(
        (
            dataset,
            row_count,
            distinct_ids,
            null_ids,
            status,
        )
    )

key_validation_df = spark.createDataFrame(
    key_checks,
    [
        "dataset",
        "row_count",
        "distinct_id_count",
        "null_or_blank_ids",
        "status",
    ],
)

display(key_validation_df)

encounters_df = source_dfs["encounters"]

encounter_timestamp_metrics = (
    encounters_df
    .select(
        F.to_timestamp("START").alias("start_ts"),
        F.to_timestamp("STOP").alias("stop_ts"),
    )
    .agg(
        F.sum(
            F.when(F.col("start_ts").isNull(), 1).otherwise(0)
        ).alias("invalid_start_rows"),
        F.sum(
            F.when(F.col("stop_ts").isNull(), 1).otherwise(0)
        ).alias("invalid_stop_rows"),
        F.min("start_ts").alias("first_encounter"),
        F.max("start_ts").alias("last_encounter"),
    )
    .first()
)

timestamp_status = (
    "PASS"
    if int(encounter_timestamp_metrics["invalid_start_rows"] or 0) == 0
    and int(encounter_timestamp_metrics["invalid_stop_rows"] or 0) == 0
    else "FAIL"
)

display(
    spark.createDataFrame(
        [(
            int(encounter_timestamp_metrics["invalid_start_rows"] or 0),
            int(encounter_timestamp_metrics["invalid_stop_rows"] or 0),
            encounter_timestamp_metrics["first_encounter"],
            encounter_timestamp_metrics["last_encounter"],
            timestamp_status,
        )],
        [
            "invalid_start_rows",
            "invalid_stop_rows",
            "first_encounter",
            "last_encounter",
            "status",
        ],
    )
)

## 11. Validate important encounter relationships

In [0]:
def count_missing_relationship(
    fact_df,
    fact_column: str,
    dimension_df,
    dimension_column: str = "Id",
) -> int:
    return (
        fact_df.alias("f")
        .join(
            dimension_df
            .select(
                F.col(dimension_column).alias("_dimension_key")
            )
            .dropDuplicates(),
            F.col(f"f.{fact_column}") == F.col("_dimension_key"),
            "left_anti",
        )
        .filter(
            F.col(fact_column).isNotNull()
            & (F.trim(F.col(fact_column).cast("string")) != "")
        )
        .count()
    )


relationship_definitions = [
    ("encounter_patient", "PATIENT", "patients"),
    ("encounter_provider", "PROVIDER", "providers"),
    ("encounter_organization", "ORGANIZATION", "organizations"),
    ("encounter_payer", "PAYER", "payers"),
]

relationship_rows = []

for check_name, fact_column, dimension_name in relationship_definitions:
    missing_count = count_missing_relationship(
        encounters_df,
        fact_column,
        source_dfs[dimension_name],
    )

    relationship_rows.append(
        (
            check_name,
            dimension_name,
            missing_count,
            "PASS" if missing_count == 0 else "WARN",
        )
    )

relationship_validation_df = spark.createDataFrame(
    relationship_rows,
    [
        "relationship",
        "dimension",
        "unmatched_non_null_rows",
        "status",
    ],
)

display(relationship_validation_df)

print(
    "Relationship mismatches are reported as WARN rather than fatal "
    "because source systems can legitimately contain unknown/retired references."
)

## 12. Profile encounter volume by month

In [0]:
monthly_encounter_profile_df = (
    encounters_df
    .withColumn(
        "encounter_start",
        F.to_timestamp("START"),
    )
    .withColumn(
        "encounter_month",
        F.date_format("encounter_start", "yyyy-MM"),
    )
    .groupBy("encounter_month")
    .agg(
        F.count("*").alias("encounter_count"),
        F.countDistinct("PATIENT").alias("unique_patients"),
    )
    .orderBy("encounter_month")
)

display(monthly_encounter_profile_df)

month_summary = (
    monthly_encounter_profile_df
    .agg(
        F.count("*").alias("observed_months"),
        F.min("encounter_month").alias("first_month"),
        F.max("encounter_month").alias("last_month"),
        F.sum("encounter_count").alias("total_encounters"),
    )
)

display(month_summary)

## 13. Stage reference/master datasets

The raw source remains under `source/csv`.  
A separate reference area is maintained for datasets that behave as master or
reference inputs for the Gold model.

`encounters.csv` remains transactional and is not copied to `reference/`.


In [0]:
REFERENCE_DATASETS = {
    "patients",
    "providers",
    "organizations",
    "payers",
    "conditions",
}

reference_stage_rows = []

for dataset in sorted(REFERENCE_DATASETS):
    filename = REQUIRED_FILES[dataset]
    source_file = f"{source_csv_path}/{filename}"
    target_file = f"{reference_path}/{filename}"

    dbutils.fs.cp(
        source_file,
        target_file,
        True,
    )

    reference_stage_rows.append(
        (
            dataset,
            filename,
            target_file,
            "STAGED",
        )
    )

reference_stage_df = spark.createDataFrame(
    reference_stage_rows,
    ["dataset", "file_name", "target_path", "status"],
)

display(reference_stage_df)

## 14. Validate staged reference data

In [0]:
staged_reference_files = {
    item.name.rstrip("/")
    for item in dbutils.fs.ls(reference_path)
}

expected_reference_files = {
    REQUIRED_FILES[name]
    for name in REFERENCE_DATASETS
}

missing_reference_files = sorted(
    expected_reference_files - staged_reference_files
)

reference_status = (
    "PASS"
    if not missing_reference_files
    else "FAIL"
)

display(
    spark.createDataFrame(
        [(
            len(expected_reference_files),
            len(
                expected_reference_files
                & staged_reference_files
            ),
            ", ".join(missing_reference_files),
            reference_status,
        )],
        [
            "expected_reference_files",
            "available_reference_files",
            "missing_reference_files",
            "status",
        ],
    )
)

if missing_reference_files:
    raise RuntimeError(
        "Reference staging validation failed: "
        + ", ".join(missing_reference_files)
    )

print("Reference staging validation passed.")

## 15. Final source-preparation validation

In [0]:
source_counts = {
    row["dataset"]: int(row["row_count"])
    for row in source_inventory_df.collect()
}

key_failures = [
    row["dataset"]
    for row in key_validation_df.collect()
    if row["status"] != "PASS"
]

final_checks = [
    ("external_volume", volume_type == "EXTERNAL"),
    ("required_source_files", len(missing_files) == 0),
    ("required_source_columns", len(column_failures) == 0),
    ("source_rows_non_empty", all(count > 0 for count in source_counts.values())),
    ("primary_keys", len(key_failures) == 0),
    ("encounter_timestamps", timestamp_status == "PASS"),
    ("reference_staging", reference_status == "PASS"),
]

final_validation_df = spark.createDataFrame(
    [
        (
            check_name,
            "PASS" if passed else "FAIL",
        )
        for check_name, passed in final_checks
    ],
    ["validation_area", "status"],
)

display(final_validation_df)

failed_checks = [
    check_name
    for check_name, passed in final_checks
    if not passed
]

if run_validation and failed_checks:
    raise RuntimeError(
        "Lab 06 source preparation failed: "
        + ", ".join(failed_checks)
    )

print("")
print("LAB 06 — EXTERNAL SOURCE PREPARATION COMPLETE")
print(f"Source volume : {volume_fqn}")
print(f"Source path   : {source_csv_path}")
print(f"Reference     : {reference_path}")
print("")
print("Next notebook : lab06_00_dev_runner")
print("Recommended   : run 01_dimensions first, then the full Gold chain.")

## Completion state

After this notebook succeeds, the following source layout is ready:

```text
/Volumes/<catalog>/<schema>/<volume_name>/
├── source/
│   └── csv/
│       ├── patients.csv
│       ├── encounters.csv
│       ├── providers.csv
│       ├── organizations.csv
│       ├── payers.csv
│       ├── conditions.csv
│       └── ...
├── reference/
│   ├── patients.csv
│   ├── providers.csv
│   ├── organizations.csv
│   ├── payers.csv
│   └── conditions.csv
└── landing/
    └── encounters/
```

The external Gold table locations are intentionally **not** created here.
`src/external_tables.py` and the subsequent Lab 06 notebooks remain responsible
for those objects.
